# PMT gain calibration versus high voltage

This notebook combines the YAML files produced by `Fit.ipynb`. Select one consistent fit model, charge-integration method, and event selection before interpreting the voltage dependence.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pmt.gain_calibration import (
    load_voltage_scan_results,
    select_voltage_scan,
    plot_gain_calibration,
    plot_fit_parameters_vs_voltage,
)

plt.rcParams['figure.dpi'] = 120

## Configuration

The defaults select the LED-window Poisson fit with the SNR ≥ 20 pulse-quality selection. Change these values to compare integration methods, models, or selection systematics.

In [ ]:
results_dir = Path(globals().get('batch_fit_results_dir', 'plots/260706/fit/fit_results'))
output_dir = Path(globals().get('batch_gain_output_dir', 'plots/260706/gain_calibration'))
output_dir.mkdir(parents=True, exist_ok=True)

fit_models = list(globals().get('batch_fit_models') or ['poisson'])
charge_methods = list(globals().get('batch_charge_methods') or ['led_window'])
selection_names = globals().get('batch_selection_names', ['pulse_quality_above_snr20'])

fit_model = fit_models[0]
integration = charge_methods[0]
selection = None if selection_names is None else list(selection_names)[0]
pmt_id = globals().get('batch_pmt_id', None)
acquisition = globals().get('batch_acquisition', None)

# Keep these runs in the all-fits table, but omit them from every plotted
# voltage scan and gain power-law fit.
excluded_voltages_V = [750.0, 775.0]


## Build and save the summary table

In [ ]:
summary = load_voltage_scan_results(results_dir)
summary_file = output_dir / 'gain_calibration_all_fits.csv'
summary.to_csv(summary_file, index=False)
calibration_summary = summary.loc[
    ~summary['voltage_V'].isin(excluded_voltages_V)
].copy()

print(f'Loaded {len(summary)} fits from {summary.voltage_V.nunique()} voltages')
print(f'Voltages currently available: {sorted(summary.voltage_V.unique())}')
print(f'Excluded from gain plots and calibrations: {excluded_voltages_V} V')
print(f'Saved summary to {summary_file}')

if pmt_id is None:
    pmt_ids = sorted(summary['pmt_id'].dropna().unique())
    if len(pmt_ids) == 1:
        pmt_id = pmt_ids[0]
    else:
        raise ValueError(f'More than one PMT id is present; set batch_pmt_id. Found: {pmt_ids}')
if acquisition is None:
    acquisitions = sorted(summary['acquisition'].dropna().unique())
    if len(acquisitions) == 1:
        acquisition = acquisitions[0]
    else:
        raise ValueError(f'More than one acquisition is present; set batch_acquisition. Found: {acquisitions}')

available_for_main = summary.loc[
    summary['pmt_id'].eq(pmt_id)
    & summary['acquisition'].eq(acquisition)
    & summary['model'].eq(fit_model)
    & summary['integration'].eq(integration)
].copy()
if available_for_main.empty:
    available_for_main = summary.loc[
        summary['pmt_id'].eq(pmt_id)
        & summary['acquisition'].eq(acquisition)
    ].copy()
    if available_for_main.empty:
        raise ValueError(f'No fits are available for {pmt_id}, {acquisition}.')
    first_choice = available_for_main[['model', 'integration']].drop_duplicates().iloc[0]
    fit_model = first_choice['model']
    integration = first_choice['integration']
    available_for_main = available_for_main.loc[
        available_for_main['model'].eq(fit_model)
        & available_for_main['integration'].eq(integration)
    ]
if selection is None or selection not in set(available_for_main['selection']):
    preferred = ['led_timing_above_snr10', 'led_timing_above_snr8', 'led_timing_above_snr12', 'led_timing_above_snr15', 'pulse_quality_above_snr20', 'pulse_quality_above_snr15', 'no_peak_cuts']
    available_selections = list(available_for_main['selection'].drop_duplicates())
    selection = next(
        (candidate for candidate in preferred if candidate in available_selections),
        sorted(available_selections)[0],
    )

print(f'Using PMT/acquisition filter: {pmt_id}, {acquisition}')
print(f'Main gain plot: {fit_model}, {integration}, {selection}')
display(summary.head())


In [ ]:
scan = select_voltage_scan(
    calibration_summary,
    model=fit_model,
    integration=integration,
    selection=selection,
    pmt_id=pmt_id,
    acquisition=acquisition,
)

columns = [
    'voltage_V', 'q1_mV_ns', 'q1_mV_ns_error',
    'gain', 'gain_error', 'q1_mV_ns_at_bound', 'q0_mV_ns', 'sigma0_mV_ns',
    'sigma1_mV_ns', 'mu_pe', 'reduced_chi2',
]
display(scan[[column for column in columns if column in scan.columns]])

## SPE charge and gain curve

The conventional PMT calibration is represented as $G(V)=G_{ref}(V/V_{ref})^k$. At least three voltages are recommended before quoting uncertainties on the power-law parameters.

In [ ]:
fig, axes, calibration = plot_gain_calibration(scan, fit_power_law=True)
fig.suptitle(f'{pmt_id}: {fit_model}, {integration}, {selection}', y=1.03)
fig.savefig(output_dir / 'gain_curve.png', bbox_inches='tight')

if calibration is not None:
    display(pd.Series(calibration, name='gain calibration'))

## Other fit parameters versus voltage

These plots are diagnostics. In particular, strong changes in pedestal width, occupancy, reduced chi-square, or SPE resolution can reveal unstable fits or changing acquisition conditions.

In [ ]:
fig, axes = plot_fit_parameters_vs_voltage(scan)
fig.suptitle(f'{pmt_id}: fit diagnostics versus voltage', y=1.01)
fig.savefig(output_dir / 'fit_parameters_vs_voltage.png', bbox_inches='tight')

## Comparing analysis choices

The batch below creates one graph and one gain calibration for every exact `(model, integration, selection)` combination. Each curve therefore compares voltages only; analysis criteria are never mixed within a curve.

In [ ]:
# Keep PMT and acquisition fixed, then enumerate every analysis choice that
# is actually present in the fit-result files.
available = calibration_summary.loc[
    calibration_summary['pmt_id'].eq(pmt_id)
    & calibration_summary['acquisition'].eq(acquisition)
].copy()
analysis_choices = (
    available[['model', 'integration', 'selection']]
    .drop_duplicates()
    .sort_values(['model', 'integration', 'selection'])
)

calibration_rows = []
consistent_scans = {}
calibrations_by_analysis = {}
for choice in analysis_choices.itertuples(index=False):
    consistent_scan = select_voltage_scan(
        calibration_summary,
        model=choice.model,
        integration=choice.integration,
        selection=choice.selection,
        pmt_id=pmt_id,
        acquisition=acquisition,
    )
    analysis_name = f'{choice.model}__{choice.integration}__{choice.selection}'
    title = (
        f'{pmt_id}: {choice.model}; {choice.integration}; '
        f'{choice.selection}'
    )

    fig, axes, choice_calibration = plot_gain_calibration(
        consistent_scan, fit_power_law=True
    )
    fig.suptitle(title, y=1.03)
    fig.savefig(
        output_dir / f'gain_curve__{analysis_name}.png',
        bbox_inches='tight',
    )
    plt.close(fig)

    row = {
        'pmt_id': pmt_id,
        'acquisition': acquisition,
        'model': choice.model,
        'integration': choice.integration,
        'selection': choice.selection,
        'n_voltages_available': consistent_scan['voltage_V'].nunique(),
    }
    if choice_calibration is not None:
        row.update(choice_calibration)
        row['status'] = 'fit'
    else:
        row['status'] = 'not fit: fewer than two usable voltages'
    calibration_rows.append(row)
    consistent_scans[analysis_name] = consistent_scan
    calibrations_by_analysis[analysis_name] = choice_calibration

all_calibrations = pd.DataFrame(calibration_rows)
all_calibrations.to_csv(output_dir / 'gain_calibrations_by_analysis.csv', index=False)
print(f'Saved {len(all_calibrations)} separate analysis comparisons to {output_dir}')
display(all_calibrations)

## Gain-calibration grids

One compact grid is produced per SPE fit model. Every panel keeps one integration and selection criterion fixed while comparing voltages.

In [ ]:
grid_ncols = 4
grid_figures = {}

for grid_model in analysis_choices['model'].drop_duplicates():
    model_choices = analysis_choices.loc[analysis_choices['model'].eq(grid_model)]
    n_panels = len(model_choices)
    nrows = int(np.ceil(n_panels / grid_ncols))
    fig, axes = plt.subplots(
        nrows, grid_ncols,
        figsize=(4.2 * grid_ncols, 3.4 * nrows),
        sharex=True, sharey=True, squeeze=False,
    )

    for ax, choice in zip(axes.ravel(), model_choices.itertuples(index=False)):
        analysis_name = f'{choice.model}__{choice.integration}__{choice.selection}'
        choice_scan = consistent_scans[analysis_name]
        choice_calibration = calibrations_by_analysis[analysis_name]
        bound_hits = choice_scan.get(
            'q1_mV_ns_at_bound',
            pd.Series(False, index=choice_scan.index),
        ).fillna(False)

        ax.errorbar(
            choice_scan['voltage_V'], choice_scan['gain'],
            yerr=choice_scan.get('gain_error'),
            fmt='o', capsize=2.5, markersize=4, label='SPE fit',
        )
        if bound_hits.any():
            ax.scatter(
                choice_scan.loc[bound_hits, 'voltage_V'],
                choice_scan.loc[bound_hits, 'gain'],
                marker='x', s=55, color='tab:red', zorder=5,
                label='Q1 at bound',
            )
        if choice_calibration is not None:
            voltage_curve = np.linspace(
                choice_scan['voltage_V'].min(),
                choice_scan['voltage_V'].max(), 250,
            )
            gain_curve = choice_calibration['gain_at_reference'] * (
                voltage_curve / choice_calibration['reference_voltage_V']
            ) ** choice_calibration['exponent']
            ax.plot(
                voltage_curve, gain_curve,
                label=rf"$G \propto V^{{{choice_calibration['exponent']:.2f}}}$",
            )

        selection_label = choice.selection.replace('pulse_quality_above_', '').replace('_', ' ')
        integration_label = choice.integration.replace('_', ' ')
        ax.set_title(f'{integration_label} | {selection_label}', fontsize=9)
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7)

    for ax in axes.ravel()[n_panels:]:
        ax.set_visible(False)
    fig.supxlabel('PMT voltage [V]')
    fig.supylabel('Gain')
    fig.suptitle(f'{pmt_id}: {grid_model} gain calibrations', fontsize=14)
    fig.tight_layout(rect=(0.02, 0.02, 1, 0.96))
    fig.savefig(
        output_dir / f'gain_curve_grid__{grid_model}.png',
        dpi=160, bbox_inches='tight',
    )
    grid_figures[grid_model] = fig

print(f'Saved {len(grid_figures)} gain-calibration grids to {output_dir}')